# 7. Naive Bayes

Naive Bayes classifiers are a family of probabilistic classifiers built on **Bayes' Theorem** with a strong (naive) assumption of independence between features. Despite their simplicity, they work surprisingly well for many real-world problems — especially text classification and spam detection.


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.datasets import load_iris, load_wine, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print('All imports successful!')


All imports successful!


<hr>


## 7.1 Introduction to Naive Bayes

Naive Bayes is built on the foundation of **Bayes' Theorem**, which describes the probability of an event given prior knowledge of conditions that might be related.


#### 7.1.1 Bayes Theorem

The theorem is mathematically expressed as:

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Where:
- $P(A|B)$ is the **posterior** probability of class A given predictor B
- $P(B|A)$ is the **likelihood** of predictor B given class A
- $P(A)$ is the **prior** probability of class A
- $P(B)$ is the **evidence** or marginal probability


#### 7.1.2 Conditional Probability Intuition

Imagine you have a basket of fruits: 60% apples and 40% oranges. Among apples, 30% are red; among oranges, 80% are red. If you pick a red fruit, what is the probability it is an apple? That's exactly what Bayes' Theorem computes!


In [2]:
# Bayes Theorem example: fruit basket
p_apple = 0.60       # Prior: 60% apples
p_orange = 0.40      # Prior: 40% oranges
p_red_given_apple = 0.30    # Likelihood: 30% of apples are red
p_red_given_orange = 0.80   # Likelihood: 80% of oranges are red

# Evidence: P(red)
p_red = p_red_given_apple * p_apple + p_red_given_orange * p_orange

# Posterior: P(apple | red)
p_apple_given_red = (p_red_given_apple * p_apple) / p_red

print('P(apple | red) = %.4f' % p_apple_given_red)
print('P(orange | red) = %.4f' % (1 - p_apple_given_red))


P(apple | red) = 0.3600
P(orange | red) = 0.6400


<hr>


## 7.2 Gaussian Naive Bayes

**Gaussian Naive Bayes** assumes that the likelihood of continuous features follows a **Gaussian (normal) distribution**. It is the most common variant for real-valued data.


#### 7.2.1 Load Dataset & Train

We'll use the **Iris dataset** — a classic multi-class classification problem with 4 continuous features.


In [3]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

print('Training complete!')
print('Classes: %s' % str(iris.target_names))
print('Feature names: %s' % str(iris.feature_names))


Training complete!
Classes: ['setosa' 'versicolor' 'virginica']
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [4]:
accuracy = accuracy_score(y_test, y_pred)
print('GaussianNB Accuracy: %.4f' % accuracy)
print('\nPredictions vs Actual (first 10):')
for i in range(min(10, len(y_test))):
    print('  Sample %d: Predicted=%s, Actual=%s' % (
        i+1, iris.target_names[y_pred[i]], iris.target_names[y_test[i]]
    ))
print('\nConfusion Matrix:')
print(str(confusion_matrix(y_test, y_pred)))


GaussianNB Accuracy: 0.9778

Predictions vs Actual (first 10):
  Sample 1: Predicted=setosa, Actual=setosa
  Sample 2: Predicted=setosa, Actual=setosa
  Sample 3: Predicted=setosa, Actual=setosa
  Sample 4: Predicted=versicolor, Actual=versicolor
  Sample 5: Predicted=versicolor, Actual=versicolor
  Sample 6: Predicted=versicolor, Actual=versicolor
  Sample 7: Predicted=versicolor, Actual=versicolor
  Sample 8: Predicted=versicolor, Actual=versicolor
  Sample 9: Predicted=versicolor, Actual=versicolor
  Sample 10: Predicted=virginica, Actual=versicolor

Confusion Matrix:
[[19  0  0]
 [ 0 13  0]
 [ 0  1 12]]


<hr>


## 7.3 Multinomial Naive Bayes

**Multinomial Naive Bayes** is designed for data where features represent **counts or frequencies** — commonly used in text classification with bag-of-words or TF-IDF vectors.


#### 7.3.1 Text-Like Data Example

We simulate a simple document-term count matrix to demonstrate.


In [5]:
# Simulated document-term count matrix (5 docs, 4 words)
X_counts = np.array([
    [3, 1, 0, 0],   # Doc 1: sports
    [2, 0, 0, 1],   # Doc 2: sports
    [0, 0, 4, 1],   # Doc 3: politics
    [1, 0, 3, 0],   # Doc 4: politics
    [0, 2, 1, 0],   # Doc 5: mixed
])
y_counts = np.array([0, 0, 1, 1, 0])  # 0 = sports, 1 = politics

mnb = MultinomialNB()
mnb.fit(X_counts, y_counts)

# Predict on new document: [2, 0, 1, 0]
new_doc = np.array([[2, 0, 1, 0]])
pred = mnb.predict(new_doc)
probs = mnb.predict_proba(new_doc)

print('MultinomialNB trained on %d documents.' % X_counts.shape[0])
print('New document prediction: Class %d' % pred[0])
print('Probabilities: Sports=%.4f, Politics=%.4f' % (probs[0][0], probs[0][1]))


MultinomialNB trained on 5 documents.
New document prediction: Class 0
Probabilities: Sports=0.8892, Politics=0.1108


<hr>


## 7.4 Bernoulli Naive Bayes

**Bernoulli Naive Bayes** is designed for **binary / boolean features** (e.g., presence or absence of a word in a document). Each feature is modeled as a Bernoulli random variable.


#### 7.4.1 Binary Feature Example

We use a simple binary matrix representing word presence (1) or absence (0).


In [6]:
# Simulated binary feature matrix (8 samples, 5 features)
X_binary = np.array([
    [1, 1, 0, 0, 0],
    [1, 0, 0, 0, 1],
    [0, 1, 1, 0, 0],
    [0, 0, 1, 1, 1],
    [1, 1, 1, 0, 0],
    [0, 0, 0, 1, 1],
    [1, 0, 1, 0, 0],
    [0, 1, 0, 1, 1],
])
y_binary = np.array([0, 0, 0, 1, 0, 1, 0, 1])

bnb = BernoulliNB()
bnb.fit(X_binary, y_binary)

# Predict a new binary sample
X_new_bin = np.array([[1, 0, 1, 0, 1]])
pred_bin = bnb.predict(X_new_bin)
probs_bin = bnb.predict_proba(X_new_bin)

print('BernoulliNB trained on %d samples.' % X_binary.shape[0])
print('Prediction for [1,0,1,0,1]: Class %d' % pred_bin[0])
print('Probability of class 0: %.4f' % probs_bin[0][0])
print('Probability of class 1: %.4f' % probs_bin[0][1])


BernoulliNB trained on 8 samples.
Prediction for [1,0,1,0,1]: Class 0
Probability of class 0: 0.6650
Probability of class 1: 0.3350


<hr>


## 7.5 Comparing NB Variants

Let's compare all three Naive Bayes variants on the same **Wine dataset** to see how they perform.


In [7]:
wine = load_wine()
X_w, y_w = wine.data, wine.target

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X_w, y_w, test_size=0.3, random_state=42
)

classifiers = {
    'GaussianNB': GaussianNB(),
    'MultinomialNB': MultinomialNB(),
    'BernoulliNB': BernoulliNB()
}

print('%-20s %-20s' % ('Classifier', 'Accuracy'))
print('%-20s %-20s' % ('-'*18, '-'*18))

results = {}
for name, clf in classifiers.items():
    clf.fit(Xw_train, yw_train)
    preds = clf.predict(Xw_test)
    acc = accuracy_score(yw_test, preds)
    results[name] = acc
    print('%-20s %.4f' % (name, acc))


Classifier           Accuracy           
------------------ ------------------
GaussianNB           0.9815
MultinomialNB        0.8704
BernoulliNB          0.4444


In [8]:
# Visual comparison
fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
accs = list(results.values())
colors = ['#2E86AB', '#A23B72', '#F18F01']
bars = ax.bar(names, accs, color=colors, edgecolor='black', linewidth=1.2)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Naive Bayes Variants Comparison on Wine Dataset', fontsize=13)
ax.set_ylim(0, 1.05)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            '%.2f' % acc, ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print('Comparison chart generated successfully!')


Comparison chart generated successfully!


<hr>


## 7.6 Pros & Cons

### Pros
- Extremely fast to train and predict — linear time complexity
- Works well with high-dimensional data (e.g., text classification)
- Performs well even with small training sets
- Handles both categorical and continuous data (with appropriate variant)
- Provides probability estimates, not just class labels

### Cons
- The **naive independence assumption** rarely holds in real data
- Struggles with correlated features
- Zero-frequency problem (handled via Laplace smoothing)
- Generally outperformed by more complex models like Random Forest or SVM


<hr>


## 7.7 Assignment

1. Load the **Digits dataset** from `sklearn.datasets.load_digits()`.
2. Split it into train (70%) and test (30%) sets.
3. Train a **Gaussian Naive Bayes** classifier and report accuracy.
4. Train a **Multinomial Naive Bayes** (binarize features with a threshold of 8) and compare accuracy.
5. Plot the first 10 test images along with their predicted and true labels.
6. Write a short paragraph explaining which variant performed better and why.